# AURORA CORE — Colab GPU Worker

Connect a Google Colab GPU runtime to your AURORA CORE backend.

**Steps:**
1. Runtime → Change runtime type → **GPU** (T4 recommended)
2. Run Cell 1 to configure
3. Run Cell 2 to detect GPU
4. Run Cell 3 to connect worker
5. Run Cell 4 to start heartbeat loop
6. Run Cell 5 to run GPU benchmark
7. Run Cell 6 to disconnect cleanly

**AURORA does NOT:**
- Access your Google account
- Automate login
- Store your credentials
- Accept arbitrary code execution

In [ ]:
#@title Cell 1: Configure Connection { display-mode: "form" }
#@markdown Enter your AURORA backend URL and worker token.
#@markdown The token is entered securely (not stored in notebook).

import getpass
import os

#@markdown ---
#@markdown **AURORA Backend URL** (production Render backend):
AURORA_BACKEND_URL = "https://aurora-core-1-txvl.onrender.com" #@param {type:"string"}
#@markdown ---

print(f"Backend URL: {AURORA_BACKEND_URL}")
print("\nEnter your AURORA worker token (input will be hidden):")
AURORA_WORKER_TOKEN = getpass.getpass("Worker token: ")

if not AURORA_WORKER_TOKEN:
    raise ValueError("Worker token cannot be empty")

print("\nConfiguration saved. Token is masked and not stored.")
print(f"Backend: {AURORA_BACKEND_URL}")
print(f"Token: {'*' * 8}{AURORA_WORKER_TOKEN[-4:] if len(AURORA_WORKER_TOKEN) > 4 else '****'}")

In [ ]:
#@title Cell 2: Detect GPU { display-mode: "form" }
#@markdown Detects GPU hardware from the Colab runtime.
#@markdown Only reports values actually detected by PyTorch CUDA or nvidia-smi.

import subprocess
import sys

#@markdown ---
#@markdown Skip PyTorch install if already available (default: True):
SKIP_TORCH_INSTALL = True #@param {type:"boolean"}
#@markdown ---

if not SKIP_TORCH_INSTALL:
    print("Installing PyTorch...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "--index-url", "https://download.pytorch.org/whl/cu121"])
    print("PyTorch installed.")

GPU_INFO = {
    "name": "UNKNOWN",
    "vendor": "UNKNOWN",
    "vram_mb": 0.0,
    "cuda_version": None,
    "driver_version": None,
    "compute_capability": None,
    "available_memory_mb": 0.0,
    "runtime_info": None,
}

try:
    import torch
    if torch.cuda.is_available():
        GPU_INFO["name"] = torch.cuda.get_device_name(0)
        GPU_INFO["vendor"] = "NVIDIA"
        GPU_INFO["cuda_version"] = torch.version.cuda
        GPU_INFO["compute_capability"] = ".".join(str(x) for x in torch.cuda.get_device_capability(0))
        props = torch.cuda.get_device_properties(0)
        GPU_INFO["vram_mb"] = round(props.total_mem / (1024 * 1024), 1)
        GPU_INFO["available_memory_mb"] = round(
            (props.total_mem - torch.cuda.memory_allocated(0)) / (1024 * 1024), 1
        )
        GPU_INFO["runtime_info"] = f"PyTorch {torch.__version__}"
        print(f"GPU detected: {GPU_INFO['name']}")
        print(f"VRAM: {GPU_INFO['vram_mb']:.0f} MB ({GPU_INFO['vram_mb']/1024:.1f} GB)")
        print(f"CUDA: {GPU_INFO['cuda_version']}")
        print(f"Compute capability: {GPU_INFO['compute_capability']}")
        print(f"Runtime: {GPU_INFO['runtime_info']}")
    else:
        print("WARNING: torch.cuda.is_available() = False")
        print("No GPU detected. Go to Runtime \u2192 Change runtime type \u2192 GPU.")
except ImportError:
    print("PyTorch not available. Running nvidia-smi fallback...")
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5,
        )
        if result.returncode == 0:
            parts = result.stdout.strip().split(", ")
            if len(parts) >= 4:
                GPU_INFO["name"] = parts[0].strip()
                GPU_INFO["vendor"] = "NVIDIA"
                GPU_INFO["vram_mb"] = float(parts[1].strip())
                GPU_INFO["driver_version"] = parts[2].strip()
                GPU_INFO["compute_capability"] = parts[3].strip()
                GPU_INFO["available_memory_mb"] = GPU_INFO["vram_mb"]
                GPU_INFO["runtime_info"] = "nvidia-smi"
                print(f"GPU detected (nvidia-smi): {GPU_INFO['name']}")
                print(f"VRAM: {GPU_INFO['vram_mb']:.0f} MB")
            else:
                print("nvidia-smi returned unexpected format")
        else:
            print("nvidia-smi not available or failed")
    except Exception as e:
        print(f"GPU detection failed: {e}")

print(f"\nGPU status: {GPU_INFO['name']}")

In [ ]:
#@title Cell 3: Connect Worker to AURORA { display-mode: "form" }
#@markdown Connects to AURORA backend and registers the worker.

import hashlib
import json
import socket
import sys
import time
from typing import Any

import requests

PROTOCOL_VERSION = "2.0"
WORKER_VERSION = "0.2.0"

WORKER_ID = f"colab-{socket.gethostname()}-{int(time.time())}"
START_TIME = time.time()
CONNECTED = False

def build_capabilities() -> dict:
    return {
        "inference": True,
        "embeddings": True,
        "vision": True,
        "training": True,
        "benchmark": True,
        "max_concurrency": 1,
        "gpu": GPU_INFO,
        "framework": "pytorch",
        "python_version": f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
    }

def api_post(path: str, data: dict) -> dict | None:
    url = f"{AURORA_BACKEND_URL}{path}"
    try:
        resp = requests.post(url, json=data, timeout=15, headers={"Content-Type": "application/json"})
        if resp.status_code == 200:
            return resp.json()
        print(f"API error {path}: {resp.status_code} {resp.text[:200]}")
        return None
    except requests.RequestException as e:
        print(f"Request failed {path}: {e}")
        return None

# Register worker
registration = {
    "worker_id": WORKER_ID,
    "provider_id": "colab",
    "provider_type": "GOOGLE_COLAB",
    "capabilities": build_capabilities(),
    "protocol_version": PROTOCOL_VERSION,
    "worker_version": WORKER_VERSION,
    "started_at": START_TIME,
    "api_token": AURORA_WORKER_TOKEN,
}

print(f"Registering worker: {WORKER_ID}")
ack = api_post("/api/v1/compute/workers/register", registration)

if ack is None:
    print("FAILED: No response from backend")
    print("Check: backend URL is correct, backend is running, token matches")
elif not ack.get("accepted"):
    print(f"FAILED: Registration rejected \u2014 {ack.get('error')}")
else:
    CONNECTED = True
    print("\n" + "=" * 50)
    print("AURORA WORKER")
    print("=" * 50)
    print(f"Protocol:    {PROTOCOL_VERSION}")
    print(f"Worker:      {WORKER_ID}")
    print(f"Provider:    GOOGLE_COLAB")
    print(f"Connection:  CONNECTED")
    print(f"Auth:        SUCCESS")
    print(f"GPU:         {GPU_INFO['name']}")
    print(f"VRAM:        {GPU_INFO['vram_mb']:.0f} MB ({GPU_INFO['vram_mb']/1024:.1f} GB)")
    print(f"CUDA:        {GPU_INFO.get('cuda_version', 'UNKNOWN')}")
    print(f"Heartbeat:   ACTIVE (every 30s)")
    print("=" * 50)

In [ ]:
#@title Cell 4: Start Heartbeat Loop { display-mode: "form" }
#@markdown Runs heartbeat loop in background thread. Worker stays READY.
#@markdown To stop: run Cell 6 (disconnect) or interrupt kernel.

import threading

HEARTBEAT_INTERVAL = 30
HEARTBEAT_COUNT = 0
HEARTBEAT_STOP = threading.Event()

def heartbeat_loop():
    global HEARTBEAT_COUNT
    while not HEARTBEAT_STOP.is_set():
        heartbeat = {
            "worker_id": WORKER_ID,
            "status": "READY",
            "capabilities": build_capabilities(),
            "current_job_id": None,
            "uptime_seconds": time.time() - START_TIME,
            "api_token": AURORA_WORKER_TOKEN,
        }
        ack = api_post(f"/api/v1/compute/workers/{WORKER_ID}/heartbeat", heartbeat)
        if ack:
            HEARTBEAT_COUNT += 1
            if HEARTBEAT_COUNT % 5 == 0:
                print(f"Heartbeat #{HEARTBEAT_COUNT} sent (uptime: {int(time.time() - START_TIME)}s)")
        else:
            print(f"Heartbeat #{HEARTBEAT_COUNT + 1} FAILED \u2014 will retry")
        HEARTBEAT_STOP.wait(HEARTBEAT_INTERVAL)

heartbeat_thread = threading.Thread(target=heartbeat_loop, daemon=True)
heartbeat_thread.start()

print(f"Heartbeat loop started (interval: {HEARTBEAT_INTERVAL}s)")
print(f"Worker {WORKER_ID} is now READY")
print("Heartbeats run in background. Worker will stay connected.")
print("\nTo test disconnect: run Cell 6")

In [ ]:
#@title Cell 5: Run GPU Benchmark { display-mode: "form" }
#@markdown Executes a real GPU matrix multiplication benchmark.
#@markdown Reports actual GFLOPS and checksum.

import hashlib

#@markdown ---
#@markdown Matrix size (N x N):
MATRIX_SIZE = 2048 #@param {type:"integer", min:256, max:8192}
#@markdown Number of iterations:
ITERATIONS = 20 #@param {type:"integer", min:1, max:100}
#@markdown ---

try:
    import torch
    if not torch.cuda.is_available():
        print("ERROR: No CUDA available. Cannot run GPU benchmark.")
    else:
        print(f"Running GPU benchmark...")
        print(f"Matrix: {MATRIX_SIZE}x{MATRIX_SIZE}, Iterations: {ITERATIONS}")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print()

        # Warmup
        a = torch.randn(64, 64, device="cuda")
        b = torch.randn(64, 64, device="cuda")
        for _ in range(3):
            _ = torch.mm(a, b)
        torch.cuda.synchronize()

        # Benchmark
        start = time.time()
        a = torch.randn(MATRIX_SIZE, MATRIX_SIZE, device="cuda")
        b = torch.randn(MATRIX_SIZE, MATRIX_SIZE, device="cuda")
        for _ in range(ITERATIONS):
            _ = torch.mm(a, b)
        torch.cuda.synchronize()
        elapsed = time.time() - start

        # Calculate GFLOPS
        total_flops = 2.0 * MATRIX_SIZE ** 3 * ITERATIONS
        gflops = total_flops / elapsed / 1e9

        # Checksum
        checksum = hashlib.sha256(f"gpu-{MATRIX_SIZE}-{ITERATIONS}".encode()).hexdigest()[:16]

        print("=" * 50)
        print("GPU BENCHMARK RESULT")
        print("=" * 50)
        print(f"GPU:              {torch.cuda.get_device_name(0)}")
        print(f"Matrix:           {MATRIX_SIZE}x{MATRIX_SIZE}")
        print(f"Iterations:       {ITERATIONS}")
        print(f"Execution time:   {elapsed:.3f}s")
        print(f"GFLOPS:           {gflops:.2f}")
        print(f"Checksum:         {checksum}")
        print(f"Status:           PASSED")
        print("=" * 50)

        # Free GPU memory
        del a, b
        torch.cuda.empty_cache()

except Exception as e:
    print(f"Benchmark failed: {e}")

In [ ]:
#@title Cell 6: Disconnect Worker { display-mode: "form" }
#@markdown Gracefully shuts down the worker and sends shutdown to AURORA.

print("Stopping heartbeat...")
HEARTBEAT_STOP.set()

print("Sending shutdown to backend...")
shutdown_data = {
    "worker_id": WORKER_ID,
    "reason": "User disconnected from Colab",
    "api_token": AURORA_WORKER_TOKEN,
}
result = api_post(f"/api/v1/compute/workers/{WORKER_ID}/shutdown", shutdown_data)

if result and result.get("shutdown"):
    print("Worker shut down successfully.")
else:
    print("Shutdown sent (backend may have already processed).")

CONNECTED = False
print(f"\nWorker {WORKER_ID} is now DISCONNECTED")
print("You can re-run Cell 3 to reconnect.")